# HEAL benchmark evals

In [37]:
# to get new pysqlite3, need to reinstall
# pip install pysqlite3-binary --force-reinstall
import pysqlite3
import sys
sys.modules["sqlite3"] = pysqlite3

In [38]:
import chromadb
import ast
import pandas as pd
import os
from FlagEmbedding import FlagModel
# set which GPU to use
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### HEAL CDE data

In [39]:
# obtained from here: https://uchicago.app.box.com/file/2056763737745?s=drlzczyp5lhwobznbgcehkzbphm5x20n
target_file = "/opt/gpudata/aartiv/heal_cde/master_cde/master_sde_v2.jsonl"
target_df = pd.read_json(target_file, lines=True)
target_df.head()

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
0,aQO0VmltMn,Address City Name City,The city or township for the address to descri...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
1,ZqxxvEdcmt,Address County Name County,A region created by territorial division by a ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
2,elEZcZ9NdL,Address Line 1,The address where a mail piece is intended to ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
3,pCW788Mqei,Address Line 2,The additional address text to describe where ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
4,w_BHatIMoA,Address Postal Code Postal Code,the address or postal information for a person...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH


In [40]:
target_df.shape

(56558, 11)

In [41]:
target_df['sde_parent_repository'].unique()

array(['NIH', 'HEAL', 'PhenX'], dtype=object)

In [42]:
heal_target_df = target_df[target_df['sde_parent_repository'] == 'HEAL']

In [43]:
heal_target_df.shape

(5072, 11)

In [44]:
heal_target_df[heal_target_df['sde_name'].str.contains('Race')]

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
27719,AI_AN\n\n\n\n\n\n,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27720,Asian,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27721,Bl_AA,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27722,HI_LA,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27723,MENA,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27724,NH_PI,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27725,White,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27726,Unkn,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27727,Not_Rep,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL
27754,AI_AN\n\n\n\n\n\n,Race and/or Ethnicity Child,Participant/subject child self declared (or pa...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Child's Race...,pediatric-demographic-cdes.xlsx,The Pediatric Demographics are a set of demogr...,Demographics,HEAL


In [45]:
heal_target_df.head(n=3)

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
22743,Crav3desire,Craving Scale desire to use,Scale describing how strong desire to use was ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No desire;;;;;;;;;9=Strong desire,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL
22744,Crav3likely,Craving Scale likelihood of use,Scale describing how likely participant is to ...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No likelihood;;;;;;;;;9=Strong likelihood,Additional Notes (Question Text): Please imagi...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL
22745,Crav3urge,Craving Scale urges,Scale describing how strong is the urge for dr...,Numeric,0;1;2;3;4;5;6;7;8;9,0 = No urge;;;;;;;;;9=Strong urge,Additional Notes (Question Text): Please rate ...,craving-scale-3-item-cde.xlsx,The Craving Scale is a 3-item measure of cravi...,Substance Use,HEAL


In [46]:
heal_target_df.iloc[0]['sde_pv_description']

'0 = No desire;;;;;;;;;9=Strong desire'

### HEAL benchmarks
- created by Brienna
- see cleaned version here https://uchicago.app.box.com/file/2076961773404

In [47]:
benchmark_file = '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv'

In [48]:
benchmark_df = pd.read_csv(benchmark_file)

In [49]:
benchmark_df.head(n=2)

,field_name,field_type,field_title,field_enumLabels,field_constraints,field_description,CDE_instrument,element_name,element_type,element_title,element_description,enumLabels,encoding,constraints.enum,standardsMappings.id,standardsMappings.source,CDE_HEAL_ID,Notes
0,dob,date,Date of birth,NaN,NaN,Contact Information: Date of birth,Demographics,BRTHDTC,string,Birth date,Birth Date of the participant.\n,NaN,NaN,NaN,NaN,NaN,https://healdata.org/mds/metadata/HDPCDE5141,NaN
1,race___0,boolean,Race: American Indian/Alaska Native,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: Race[choice=American Indi...,Demographics,AI_AN\n\n\n\n\n\n,integer,Race and/or Ethnicity,Participant/subject self declared racial and/o...,"""0"": ""not checked"",\n""1"": ""checked""",NaN,"""0"",\n""1""",C74457,CDISC,https://healdata.org/mds/metadata/HDPCDE5141,NaN


In [50]:
benchmark_df.shape

(48, 18)

In [51]:
benchmark_file = '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476.csv'

In [52]:
benchmark_df = pd.read_csv(benchmark_file)

In [53]:
benchmark_df.shape

(1443, 18)

In [54]:
benchmark_df[~benchmark_df['element_name'].isnull()].shape

(90, 18)

In [55]:
benchmark_df[~benchmark_df['element_name'].isnull()].head()

,field_name,field_type,field_title,field_enumLabels,field_constraints,field_description,CDE_instrument,element_name,element_type,element_title,element_description,enumLabels,encoding,constraints.enum,standardsMappings.id or ID,standardsMappings.source,CDE_HEAL_ID,Notes
25,consent_dob,date,Child's date of birth:,NaN,NaN,Demographics: Child's date of birth:,Demographics,BRTHDTC,string,Birth date (child),"Date (and time, if applicable and known) the p...",NaN,NaN,NaN,C68615,CDISC,https://healdata.org/mds/metadata/HDPCDE5131,NaN
29,consent_genident,integer,What is the child's gender identity:,"{'1': 'Male', '2': 'Female', '3': 'Unknown', '...","['1', '2', '3', '4']",Demographics: What is the child's gender ident...,Demographics,GENIDENT,integer,Gender identification type child,Self-reported (or parent reported) participant...,"""1"": ""Male"",\n""2"": ""Female"",\n""3"": ""Unknown"",\...",NaN,"""1"",\n""2"",\n""3"",\n""4""",C158277,CDISC,https://healdata.org/mds/metadata/HDPCDE5131,NaN
30,consent_genideoth,string,No field label for this variable,NaN,NaN,Demographics:,Demographics,GENIDENTOTH,string,Gender identification other type text child,The free-text field related to Gender idenitif...,NaN,NaN,NaN,C158277,CDISC,NaN,NaN
64,consent_race_child___1,boolean,Consent_Race_Child: White,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: What is the child's race ...,Demographics,White,integer,Race and/or Ethnicity Child,Participant/subject self declared racial and/o...,"""0"": ""not checked"",\n""1"": ""checked""",NaN,"""0"",\n""1""",C74457,NCIT,https://healdata.org/mds/metadata/HDPCDE5131,NaN
65,consent_race_child___2,boolean,Consent_Race_Child: Black/African American,"{'0': 'Unchecked', '1': 'Checked'}","['0', '1']",Contact Information: What is the child's race ...,Demographics,Bl_AA,integer,Race and/or Ethnicity Child,Participant/subject child self declared (or pa...,"""0"": ""not checked"",\n""1"": ""checked""",NaN,"""0"",\n""1""",C74457,NCIT,https://healdata.org/mds/metadata/HDPCDE5131,NaN


In [56]:
filtered_df = benchmark_df[~benchmark_df['element_name'].isnull()]
filtered_df.to_csv('/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv')

### format name, desc and values 
- similar to training

In [57]:
heal_target_df[heal_target_df['sde_id'] == 'Asian']

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
27720,Asian,Race and/or Ethnicity,Participant/subject self declared racial and/o...,Numeric Values,0;1\n,0=not checked; 1=checked,Additional Notes (Question Text): Race and/or ...,adult-demographics-cdes.xlsx,The Adult Demographics are a set of demographi...,Demographics,HEAL


In [58]:
# one way of representing name, description and values, experiment with other representations (TO-DO)
# does not have sde_additional_info, could be added and reformatted (TO-DO)
def preprocess_variable(row):
    name = row.get(f'sde_name') or "[NO_NAME]"
    desc = row.get(f'sde_description') or "[NO_DESC]"
    raw_values = row.get(f'sde_permissible_values')

    if raw_values:
        values_list = [v.strip() for v in raw_values.split(";") if v.strip()]
        value_descriptions = row.get(f'sde_pv_description')
        if value_descriptions:
            values_descriptions_list = [v.strip() for v in value_descriptions.split(";") if v.strip()]
        else:
            values_descriptions_list = []
        value = ", ".join(values_list + values_descriptions_list) if values_list else "[NO_VALUE]"
    else:
        value = "[NO_VALUE]"

    return f"{name} {desc} {value}"

In [59]:
heal_target_df["name_desc_val"] = heal_target_df.apply(preprocess_variable, axis=1)

/tmp/ipykernel_1915718/1442155599.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  heal_target_df["name_desc_val"] = heal_target_df.apply(preprocess_variable, axis=1)


### Create embeddings

In [60]:
def init_emb_model(emb_model):
    model = FlagModel(emb_model, use_fp16=True)
    models = {
        "name_desc_val": model
    }
    return models

In [61]:
def create_target_embeddings(models):
    embeddings = {
        "name_desc_val": models["name_desc_val"].encode(heal_target_df["name_desc_val"].tolist())
    }
    return embeddings

In [62]:
def create_chroma_client_and_add_to_collection(emb_model, embeddings):
    client = chromadb.Client()
    emb_model_name_formatted = emb_model.replace('/','-')
    name = 'heal_name_desc_val' + emb_model_name_formatted
    try:
        # delete collection if already exists
        client.delete_collection(name=name)
    except Exception:
        print('collection does not exist, do nothing')
    
    collection = client.create_collection(name, metadata={"hnsw:space": "cosine"})
    batch_size = 5000
    for i in range(0, heal_target_df.shape[0], batch_size):
        print(f'adding records to collection, from {i} to {i+batch_size}')
        # just supply a list of embeddings and metadata to chroma
        # see https://docs.trychroma.com/docs/collections/add-data
        collection.add(
            embeddings=embeddings["name_desc_val"][i:i+batch_size].tolist(),
            ids=[str(id) for id in heal_target_df.index[i:i+batch_size].tolist()]
        )
    return collection


## evaluation

### HEAL benchmark

In [63]:
def preprocess_variable_benchmark(row):
    name = row.get(f'field_title') or "[NO_NAME]"
    desc = row.get(f'field_description') or "[NO_DESC]"
    raw_values = row.get(f'field_enumLabels')

    if isinstance(raw_values, dict):
        value_labels = ','.join([f'{k}={v}' for k, v in raw_values.items()])
        value_constraints = ','.join(row.get(f'field_constraints'))
        value = ';'.join([value_constraints, value_labels])
        
    else:
        value = "[NO_VALUE]"

    return f"{name} {desc} {value}"

In [64]:
def get_top_k(query_combined_emb, collection, k):
    results = collection.query(
        query_embeddings=query_combined_emb,
        n_results=k
    )
    return results


def return_top_k_results(row, models, collection, k):
    query_combined_emb = models["name_desc_val"].encode(row['name_desc_val'])
    results = get_top_k(query_combined_emb, collection, k)
    return results


def index_to_name(index_list):
    name_list = []
    for index in index_list:
        # TO-DO
        # return both sde_id and sde_name for evals
        # e.g. Asian and Race and/or Ethnicity, instead of just Race and/or Ethnicity
        name = heal_target_df.loc[int(index)]["sde_name"]
        name_list.append(name)
    return name_list


def format_results(results):
    # TO-DO add sde_id
    formatted_results_df = pd.DataFrame({
        'ids': results['ids'][0],
        'distances': results['distances'][0]
    })
    formatted_results_df.sort_values(by='distances', ascending=True, inplace=True)
    formatted_results_df['names'] = index_to_name(formatted_results_df['ids'])
    return pd.Series([
        formatted_results_df['ids'].to_list(),
        formatted_results_df['distances'].to_list(),
        formatted_results_df['names'].to_list()
    ])


def process_benchmark(benchmark_name, emb_model):
    print(f'processing {benchmark_name}')
    df = pd.read_csv(benchmark_name)
    print(f'init {emb_model}')
    models = init_emb_model(emb_model)
    print('create target embeddings')
    embeddings = create_target_embeddings(models)
    collection = create_chroma_client_and_add_to_collection(emb_model, embeddings)
    df['field_enumLabels'] = df['field_enumLabels'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df['field_constraints'] = df['field_constraints'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df['name_desc_val'] = df.apply(preprocess_variable_benchmark, axis=1)
    
    # embed query variables -- var name, var desc and return top_k
    # by searching CDE embeddings
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        col_name = f'top_{k}_results'
        df[col_name] = df.apply(lambda x: return_top_k_results(x, models, collection, k=k), axis=1)
        # format results
        output_cols = f'top_{k}_ids,top_{k}_distances,top_{k}_names'.split(',')
        df[output_cols] = df[col_name].apply(
            lambda x: format_results(x)
        )
    print('returning top k results')
    return df

def calculate_acc(truth, pred):
  correct = 0
  for t, p_list_of_names in zip(truth, pred):
      if t in p_list_of_names:
          correct += 1
  return correct / len(truth)


def run_evals(df, benchmark_name, embedding_model):
    # print('calculating metrics')
    evals = {}
    row_index = []
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        evals[f'accuracy_{k}'] = []

    row_index.append(f'{benchmark_name}_{df.shape[0]}_{embedding_model}')
    for k in top_k_list:
        col_name = f'top_{k}_names'
        top_k_names_list = df[col_name].to_list()
        # only considering "sde_name => element_title" at the moment
        # TO-DO: consider sde_id, sde_name => element_name, element_title
        # e.g. Asian, Race and/or Ethnicity
        truth_list = df['element_title'].to_list()
        accuracy = calculate_acc(truth_list, top_k_names_list)
        evals[f'accuracy_{k}'].append(accuracy)

    # print('returning metrics')
    return pd.DataFrame(evals, index=row_index)


# normalize spaces etc for truth comparison
def normalize(s):
    return " ".join(s.split())


def run_evals_per_row(row, k):
    # TO-DO: adjust depending on element_name, element_title
    col_name = f'top_{k}_names'
    top_k_names_list = row[col_name]
    truth = row['element_title']
    # print(f'truth: {truth}')
    # print(f'top_k_names: {top_k_names_list}')
    try:
        is_match = normalize(truth) in [normalize(x) for x in top_k_names_list]
    except Exception as e:
        # print(f'in exception! truth: {truth}, top_k_names_list: {top_k_names_list}')
        # print(f'original row: {row}')
        is_match = truth in top_k_names_list
    return is_match



def get_metrics_per_row(df):
    top_k_list = [5]
    for k in top_k_list:
        output_cols = f'is_match_in_top_{k}_name_desc'
        df[output_cols] = df.apply(lambda x: run_evals_per_row(x, k), axis=1)
    return df

In [65]:
embedding_model_list = ['BAAI/bge-large-en-v1.5', 'uc-ctds/bge-large-en-v1.5-bio-mapping']
benchmark_file_list = ['/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv',
                       '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv']
metrics_per_row_results = pd.concat([
    get_metrics_per_row(df=process_benchmark(f, emb_model))
    for emb_model in embedding_model_list
    for f in benchmark_file_list
])

processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
init BAAI/bge-large-en-v1.5
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 77.44it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.99it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv
init BAAI/bge-large-en-v1.5
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 74.48it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.98it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
init uc-ctds/bge-large-en-v1.5-bio-mapping
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 75.72it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.97it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv
init uc-ctds/bge-large-en-v1.5-bio-mapping
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 75.06it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.97it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results


In [66]:
metrics_per_row_results.to_csv('bge_large.ctds_model.heal_benchmark.metrics_per_row.csv')

In [67]:
embedding_model_list = ['BAAI/bge-large-en-v1.5', 'uc-ctds/bge-large-en-v1.5-bio-mapping']
benchmark_file_list = ['/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv',
                       '/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv']
results = pd.concat([
    run_evals(df=process_benchmark(f, emb_model), benchmark_name=f, embedding_model=emb_model)
    for f in benchmark_file_list
    for emb_model in embedding_model_list
])

processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
init BAAI/bge-large-en-v1.5
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 64.34it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.96it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv
init uc-ctds/bge-large-en-v1.5-bio-mapping
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 75.37it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.97it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv
init BAAI/bge-large-en-v1.5
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 76.39it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.96it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results
processing /opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv
init uc-ctds/bge-large-en-v1.5-bio-mapping
create target embeddings


pre tokenize: 100%|██████████| 20/20 [00:00<00:00, 78.26it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 20/20 [00:06<00:00,  2.97it/s]


adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
returning top k results


In [68]:
results

,accuracy_1,accuracy_5,accuracy_10
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv_48_BAAI/bge-large-en-v1.5,0.479167,0.812500,0.854167
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP00895-FILTERED.csv_48_uc-ctds/bge-large-en-v1.5-bio-mapping,0.395833,0.791667,0.895833
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv_90_BAAI/bge-large-en-v1.5,0.300000,0.822222,0.888889
/opt/gpudata/aartiv/heal_cde/benchmarks/HEAL_CDE_Mappings-HDP01476-FILTERED.csv_90_uc-ctds/bge-large-en-v1.5-bio-mapping,0.588889,0.855556,0.888889


In [69]:
results.to_csv('bge_large.ctds_model.heal_benchmark.results.csv')

# TO-DO
- experiment with a few other embedding models (MTEB leaderboard https://huggingface.co/spaces/mteb/leaderboard) (all-miniLM-L6-v2, e5-large-v2,  )
- add sde_id and sde_name for evals (mapps to element_name, element_title in benchmark)
- experiment with other ways of representing values for increasing accuracy of ctds model
- once best accuracy obtained, output rows where right answer was not achieved by ctds model with suggestions on what type ot data to add for training
- experiment with additional benchmarks

In [ ]:
# ctds model is 335M params
embedder = FlagModel('uc-ctds/bge-large-en-v1.5-bio-mapping', use_fp16=True)
model = embedder.model
# count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 335,141,888
